# Kayıp Fonksiyonları

Bu alıştırmada, Kayıp fonksiyonlarının `LinearRegression` modeli üzerindeki etkilerini karşılaştıracaksınız.

👇 Bu zorluk için kullanmak üzere bir CSV dosyası indirelim ve onu bir DataFrame'e dönüştürelim

In [1]:
import pandas as pd

data = pd.read_csv("https://d32aokrjazspmn.cloudfront.net/materials/loss_functions_dataset.csv")
data.sample(5)

,Relative Compactness,Surface Area,Wall Area,Roof Area,Overall Height,Glazing Area,Average Temperature
330,0.64,784.0,343.0,220.5,3.5,0.25,19.305
745,0.74,686.0,245.0,220.5,3.5,0.40,15.405
22,0.76,661.5,416.5,122.5,7.0,0.00,27.280
603,0.74,686.0,245.0,220.5,3.5,0.40,15.765
373,0.66,759.5,318.5,220.5,3.5,0.25,14.390


🎯 Göreviniz, tasarımına göre bir seranın içindeki ortalama sıcaklığı tahmin etmektir. Sıcaklık tahminleriniz, her bir bitki için iklim ihtiyaçlarına göre uygun sera tasarımını seçmenize yardımcı olacaktır.

🌿 Bitkilerin küçük sıcaklık değişimlerini kaldırabildiğini, ancak sıcaklık değişimleri arttıkça katlanarak daha duyarlı hale geldiğini biliyorsunuz.

## 1. Teori

❓ Teorik olarak, bitkileri öldürme riskini sınırlamak için modelinizi hangi Kayıp fonksiyonu üzerinde eğitirsiniz?

<details>
<summary> 🆘 Cevap </summary>
    
Teorik olarak, Ortalama Kare Hata (MSE) Kayıp fonksiyonunu kullanırsınız. Bu, aykırı tahminleri cezalandırır ve modelinizin büyük hatalar yapmasını engeller. Bu, daha küçük sıcaklık değişimleri ve bitkiler için daha düşük risk sağlayacaktır.

</details>

Ortalama Kare Hata

## 2. Uygulama

### 2.1 Ön İşleme

❓ Özellikleri standartlaştırın

In [11]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import SGDRegressor
from sklearn.model_selection import cross_validate

In [3]:
X=data.drop(columns="Average Temperature" , axis=1)
y=data["Average Temperature"]

In [4]:
X_rescaled=MinMaxScaler().fit_transform(X)
X_rescaled

array([[1.        , 0.        , 0.28571429, 0.        , 1.        ,
        0.        ],
       [1.        , 0.        , 0.28571429, 0.        , 1.        ,
        0.        ],
       [1.        , 0.        , 0.28571429, 0.        , 1.        ,
        0.        ],
       ...,
       [0.        , 1.        , 0.71428571, 1.        , 0.        ,
        1.        ],
       [0.        , 1.        , 0.71428571, 1.        , 0.        ,
        1.        ],
       [0.        , 1.        , 0.71428571, 1.        , 0.        ,
        1.        ]])

### 2.2 Modelleme

Bu bölümde, farklı Kayıp fonksiyonları üzerinde optimize edilmiş modelleri değerlendirerek teoriyi doğrulayacaksınız.

### En Küçük Kareler (MSE) Kaybı

❓ **En Küçük Kareler Kaybı** (MSE) üzerinde **Stokastik Gradyan İnişi** (SGD) ile optimize edilmiş bir Doğrusal Regresyon modelini **10-Katlı Çapraz doğrula**

In [15]:
clf=SGDRegressor()
scores=cross_validate(clf, X_rescaled, y, scoring="neg_mean_squared_error",cv=10)

scores

{'fit_time': array([0.01209712, 0.01570201, 0.01247191, 0.00818229, 0.0087595 ,
        0.00522065, 0.01097107, 0.01081371, 0.00834298, 0.00930023]),
 'score_time': array([0.00205922, 0.00291061, 0.0015831 , 0.00085211, 0.00105524,
        0.00068617, 0.00087786, 0.00088334, 0.00081706, 0.00089097]),
 'test_score': array([-18.73763729,  -6.89442007,  -9.92450945, -10.67107933,
         -6.51352028, -10.35404222,  -6.97857007,  -9.57490092,
        -10.03673657,  -6.34310434])}

❓ Hesaplayın:
- Ortalama çapraz doğrulanmış R2 skoru ve bunu `r2` değişkeninde kaydedin
- Tüm katlarınızın °C cinsinden en büyük tek tahmin hatasını hesaplayın ve `max_error_celsius` değişkeninde kaydedin

(İpucu: `max_error` sklearn'de kabul edilen bir puanlama metriğidir)

In [18]:
scoresr2=cross_validate(clf, X_rescaled, y, scoring="r2",cv=10)
r2=scoresr2[ 'test_score'].mean()

In [19]:
r2

0.891981669066559

In [21]:
scoremax=cross_validate(clf, X_rescaled, y, scoring="neg_max_error",cv=10)
max_error_celsius=scoremax[ 'test_score'].mean()
max_error_celsius

-9.222906364301869

### Ortalama Mutlak Hata (MAE) Kaybı

Peki modelimizi MAE üzerinde optimize edersek ne olur?

❓ **MAE** Kaybı üzerinde **Stokastik Gradyan İnişi** (SGD) ile optimize edilmiş bir Doğrusal Regresyon modelini **10-Katlı Çapraz doğrula**

<details>
<summary>💡 İpuçları</summary>

- MAE kaybı `SGDRegressor`'da doğrudan belirtilemez. Doğru parametreleri ayarlayarak tasarlanması gerekir

</details>

In [22]:
clf=SGDRegressor(loss="epsilon_insensitive" , epsilon=0)
scores_mae=cross_validate(clf, X_rescaled, y, scoring="neg_mean_absolute_error",cv=10)

scores_mae

{'fit_time': array([0.008219  , 0.0065093 , 0.01054525, 0.00574064, 0.00600338,
        0.00548697, 0.00542045, 0.00612283, 0.00480223, 0.00565982]),
 'score_time': array([0.00161648, 0.0018959 , 0.00114632, 0.00082874, 0.00148106,
        0.00089431, 0.00123262, 0.00072598, 0.00062537, 0.00079346]),
 'test_score': array([-3.72042339, -2.00350358, -2.25861245, -2.59536862, -1.94458014,
        -2.60914857, -1.990843  , -2.47577269, -2.43359358, -1.87860599])}

❓ Hesaplayın:
- Ortalama çapraz doğrulanmış R2 skoru, bunu `r2_mae`'de saklayın
- Tüm katlarınızın en büyük tek tahmin hatasını, bunu `max_error_mae`'de saklayın

In [23]:
scores_mae_r2=cross_validate(clf, X_rescaled, y, scoring="r2",cv=10)
r2_mae=scores_mae_r2[ 'test_score'].mean()

In [24]:
score_mae_max=cross_validate(clf, X_rescaled, y, scoring="neg_max_error",cv=10)
max_error_mae=score_mae_max[ 'test_score'].mean()
max_error_mae

-11.034009485844534

In [25]:
r2_mae

0.8639551700196186

## 3. Sonuç

❓ Değerlendirdiğiniz modellerden hangisi göreviniz için en uygun görünüyor?

<details>
<summary> 🆘Cevap </summary>
    
İki model arasında ortalama çapraz doğrulanmış r2 skorları yaklaşık olarak benzer olmasına rağmen, MAE üzerinde optimize edilen modelin zaman zaman daha büyük hatalar yapma şansı daha fazladır, bu da bitkileri öldürme riskini artırır!
    
</details>

MSE

# 🏁 Kodunuzu kontrol edin ve notebook'unuzu gönderin

In [26]:
from nbresult import ChallengeResult

result = ChallengeResult(
    'loss_functions',
    r2 = r2,
    r2_mae = r2_mae,
    max_error = max_error_celsius,
    max_error_mae = max_error_mae
)

result.write()
print(result.check())


============================= test session starts ==============================
platform linux -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /home/scorp08/.pyenv/versions/3.12.9/envs/workintech/bin/python
cachedir: .pytest_cache
rootdir: /home/scorp08/workintech_projeler/S16D4-S-loss-functions/tests
plugins: anyio-4.8.0, typeguard-4.4.2
collecting ... collected 3 items

test_loss_functions.py::TestLossFunctions::test_max_error_order PASSED   [ 33%]
test_loss_functions.py::TestLossFunctions::test_r2 PASSED                [ 66%]
test_loss_functions.py::TestLossFunctions::test_r2_mae PASSED            [100%]

============================== 3 passed in 0.29s ===============================


💯 You can commit your code:

git add tests/loss_functions.pickle

git commit -m 'Completed loss_functions step'

git push origin master

